# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mab0bakar/Run-the-Starter-Notebooks/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [21]:
!git clone --depth 1 https://github.com/flyrank-bih/flyrank-ml-internship-starter.git

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 91, done.
remote: Counting objects: 100% (91/91), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 91 (delta 12), reused 53 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (91/91), 1.81 MiB | 4.43 MiB/s, done.
Resolving deltas: 100% (12/12), done.


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Task Type: Ranking

My lane is Content Refresh Prioritization. The goal is to rank webpages according to how urgently they should be reviewed by a content team. This is a ranking problem because the output is an ordered list of pages rather than a simple yes/no prediction. Pages with higher refresh priority should appear at the top of the list.

In [1]:
lane = "Content Refresh Prioritization"
task_type = "Ranking"

print("Lane:", lane)
print("Task Type:", task_type)


Lane: Content Refresh Prioritization
Task Type: Ranking


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The target is a refresh priority score. Pages that show signs of decline in performance should receive higher priority. The target is based on observed page-performance signals such as impressions, CTR, and trend direction rather than a hand-written rule.

The model's purpose is to help editors decide which pages deserve attention first.

In [30]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df["trend_direction"].value_counts()

,count
trend_direction,
down,16262
stable,5962
up,4388
new,2236
flat,1152


## 3. Success metric

*One metric you can defend. What number means 'good'?*

The primary success metric is Precision@50.

This measures how many of the top 50 pages selected by the model truly deserve review. A higher Precision@50 means the model is placing important pages near the top of the ranking.

This metric matches the real business decision because content teams can only review a limited number of pages.

In [32]:
metric = "Precision@50"
print("Chosen Metric:", metric)

Chosen Metric: Precision@50


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The unit of analysis is one webpage (content item).

Each row represents a single webpage and contains performance signals that may help determine refresh priority.

In [28]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))

df.head()

Rows: 30000
Columns: 44


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [29]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule might prioritize pages using only one signal such as impressions or CTR. However, page performance depends on multiple interacting factors including impressions, engagement, search demand, trend direction, and content characteristics.

Machine learning can combine many signals at the same time and generate a more useful ranking than a simple threshold-based rule.

The output supports a real action: deciding which pages should be refreshed first.

In [33]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df[
    [
        "search_volume",
        "impressions_90d",
        "ctr",
        "engagement_rate",
        "avg_position"
    ]
].describe()


,search_volume,impressions_90d,ctr,engagement_rate,avg_position
count,27532.000000,30000.000000,30000.000000,30000.000000,30000.00000
mean,158.882391,5200.366300,0.510733,2.534520,16.34238
std,1518.270825,16838.019547,3.279162,8.310096,15.21679
min,0.000000,1.000000,0.000000,0.000000,0.00000
25%,0.000000,81.000000,0.000000,0.000000,6.20000
50%,10.000000,731.000000,0.070000,0.000000,10.80000
75%,20.000000,3615.250000,0.290000,1.350000,22.30000
max,74000.000000,517715.000000,100.000000,100.000000,245.00000


In [34]:
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.